# Learn to Learn: Meta-Learning and MAML
### An interactive walkthrough using few-shot sine-wave regression

Standard machine learning finds a **function** `f` that maps inputs to outputs. **Meta-learning** asks a deeper question: can we *learn the learning algorithm itself*? Instead of finding `f`, we find a procedure `F` that, given a brand-new task and only a handful of examples, quickly produces a good `f`.

This notebook tells one story in four steps, all grounded in the same running example — fitting an unknown sine wave `y = A·sin(x + φ)` from only a few points ("few-shot regression"):

1. **The 3-step meta-learning framework** — reframe "learn to learn" as the familiar *function → loss → optimization* recipe, where the learnable object is a learning algorithm `F_φ`.
2. **Episodic few-shot tasks** — split each task into a *support set* (within-task training) and a *query set* (within-task testing), and sample many such tasks from a task distribution.
3. **MAML (Model-Agnostic Meta-Learning)** — learn an *initialization* that adapts to a new task in just a few gradient steps (inner loop), by optimizing post-adaptation performance (outer loop).
4. **MAML vs. multi-task pre-training** — see why an initialization optimized *to adapt* beats one optimized *to be good on average right now*.

**Learning objectives**

- Reframe meta-learning as the same three-step ML recipe, where ML finds `f` while meta-learning finds `F_φ` that produces `f`. *(→ C03, C04)*
- Construct episodic support/query tasks from a task distribution and relate them to the N-way K-shot / Omniglot benchmark. *(→ C05, C06)*
- Implement MAML's inner-loop adaptation and outer-loop meta-update. *(→ C07, C08)*
- Visualize fast adaptation from a meta-learned initialization. *(→ C09)*
- Contrast MAML with a multi-task pre-training baseline. *(→ C10, C11, C12)*

Everything runs on **CPU in a few minutes** — no dataset downloads required.

In [ ]:
# C02 — Environment setup & utilities (CPU-only, Colab-safe)
%matplotlib inline

# (1) Ensure matplotlib + ipywidgets are present (quiet install only if missing).
try:
    import matplotlib  # noqa: F401
    import ipywidgets  # noqa: F401
except ImportError:
    import subprocess, sys
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "ipywidgets", "matplotlib"],
            check=False,
        )
    except Exception as e:
        print("pip install skipped/failed, relying on present packages:", e)

# (2) Core scientific imports.
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# (3) Optional interactivity — degrade gracefully if ipywidgets is unavailable.
try:
    import ipywidgets as widgets
    from ipywidgets import interact, IntSlider, FloatSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

# (4) Force CPU and deterministic threading.
DEVICE = torch.device("cpu")
torch.set_num_threads(1)

# (5) Reproducibility helper.
def set_seed(seed: int = 0) -> None:
    """Seed Python, NumPy, and PyTorch RNGs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# (6) 1-D smoothing for learning curves; returns input unchanged if too short.
def moving_average(x, w: int = 5) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w) / w, mode="valid")

# (7) Cosmetics + readiness banner.
plt.rcParams["figure.figsize"] = (6, 4)
set_seed(0)

# Sanity checks.
assert DEVICE.type == "cpu"
assert callable(set_seed) and callable(moving_average)
assert len(moving_average(np.arange(10), 3)) == 8

print(f"Ready. torch={torch.__version__}, numpy={np.__version__}, "
      f"WIDGETS_AVAILABLE={WIDGETS_AVAILABLE} (type={type(WIDGETS_AVAILABLE).__name__}), device={DEVICE}")

## Concept 1 — Meta-learning is "learn to learn"

Every supervised ML method follows the same three-step recipe. Meta-learning uses the **exact same recipe**, just one level up.

| Step | Standard ML | Meta-learning |
|------|-------------|----------------|
| **1. Function** | A model `f_θ(x)` with parameters `θ` | A *learning algorithm* `F_φ` with meta-parameters `φ` (e.g. the **initial weights**, learning rate, or architecture). `F_φ` takes a task's training data and **outputs a model `f`**. |
| **2. Loss** | `L(θ) = Σ loss(f_θ(x), y)` over training data | `L(φ) = Σ_tasks  loss_query( f produced by F_φ )` — the *query* loss of the models `F_φ` produces, summed across **many tasks**. |
| **3. Optimization** | Find `θ* = argmin L(θ)` | Find `φ* = argmin L(φ)` |

The one-line summary:

> **Standard ML finds `f`. Meta-learning finds the `F` that finds `f`.**

What is "learnable" inside `F_φ`? Many things — the initial parameters, the optimizer, the architecture, even the data augmentation policy. In this notebook we focus on the most famous choice: **the initialization** (that's MAML). The next cell makes the pain that meta-learning solves concrete — fitting a sine wave from scratch using only a few points fails badly.

In [ ]:
# C04 — The running example: one few-shot sine task, and why from-scratch underfits.
# Defines the two shared objects reused everywhere downstream: make_sine_task + MLPRegressor.

AMP_RANGE = (0.1, 5.0)
PHASE_RANGE = (0.0, np.pi)

def make_sine_task(K: int = 10, amplitude: float = None, phase: float = None,
                   x_range: tuple = (-5.0, 5.0), n_query: int = 100, rng=None) -> dict:
    """Sample one sine-regression task: y = amplitude * sin(x + phase).

    Returns float32 tensors: support_x/support_y (K,1), query_x/query_y (n_query,1).
    query_x is sorted (dense linspace) for clean line plots.
    """
    if K < 1:
        raise ValueError("K must be >= 1")
    if n_query < 2:
        raise ValueError("n_query must be >= 2")
    gen = rng if rng is not None else np.random
    if amplitude is None:
        amplitude = float(gen.uniform(*AMP_RANGE))
    if phase is None:
        phase = float(gen.uniform(*PHASE_RANGE))

    sx = gen.uniform(x_range[0], x_range[1], size=(K, 1))
    qx = np.linspace(x_range[0], x_range[1], n_query).reshape(-1, 1)
    sy = amplitude * np.sin(sx + phase)
    qy = amplitude * np.sin(qx + phase)

    to_t = lambda a: torch.tensor(a, dtype=torch.float32, device=DEVICE)
    return {
        "support_x": to_t(sx), "support_y": to_t(sy),
        "query_x": to_t(qx), "query_y": to_t(qy),
        "amplitude": amplitude, "phase": phase,
    }

class MLPRegressor(nn.Module):
    """Standard MAML sine benchmark net: 1 -> hidden -> hidden -> 1 with ReLU."""
    def __init__(self, hidden: int = 40):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

# --- Motivating demo: fit ONE task from scratch on only K points ---
set_seed(0)
task = make_sine_task(K=10)
net = MLPRegressor().to(DEVICE)
opt = torch.optim.Adam(net.parameters(), lr=1e-2)
for _ in range(100):
    opt.zero_grad()
    loss = F.mse_loss(net(task["support_x"]), task["support_y"])
    loss.backward()
    opt.step()

with torch.no_grad():
    fit = net(task["query_x"])

plt.figure()
plt.plot(task["query_x"].numpy(), task["query_y"].numpy(), "g-", label="true curve")
plt.plot(task["query_x"].numpy(), fit.numpy(), "r--", label="from-scratch fit")
plt.scatter(task["support_x"].numpy(), task["support_y"].numpy(),
            c="k", zorder=5, label=f"{task['support_x'].shape[0]} support points")
plt.title(f"Standard ML on ONE task underfits from K points\n(A={task['amplitude']:.2f}, phase={task['phase']:.2f})")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

# Sanity checks.
assert task["support_x"].shape == (10, 1) and task["query_x"].shape == (100, 1)
assert MLPRegressor()(torch.zeros(3, 1)).shape == (3, 1)
print("make_sine_task + MLPRegressor ready. Final from-scratch support MSE:", round(loss.item(), 4))

## Concept 2 — Episodic few-shot tasks: support vs. query

Meta-learning trains on **tasks**, not single examples. Each task (an *episode*) is split two ways:

- **Support set** — the within-task *training* examples. The learner adapts to the new task using only these (K of them → "K-shot").
- **Query set** — the within-task *testing* examples. We measure how good the adapted model is here.

Across episodes we also split tasks:

- **Meta-training tasks** — many tasks sampled from a task distribution `p(T)`, used to learn `φ`.
- **Meta-testing tasks** — *brand-new, held-out* tasks used to check whether `φ` generalizes.

The classic benchmark is **N-way K-shot classification on Omniglot** (1623 handwritten characters × 20 examples each). One task = sample `N` classes, then `K` examples per class for the support set and a few more for the query set. The model must classify the query examples after seeing only `N×K` support examples.

> A subtle but important point: meta-learning legitimately *"uses testing examples during training"* — the **query** set of a meta-training task acts as that task's test, and its loss is exactly what the outer loop optimizes. The truly held-out check is the *meta-test tasks*, which the learner never sees.

Our lightweight analogue: each **sine task** has a different `(A, φ)`. The K sampled points are the support set; the dense true curve is the query set. The next cell builds a distribution over such tasks.

In [ ]:
# C06 — Episodic task distribution + interactive exploration.

class TaskDistribution:
    """A distribution p(T) over sine tasks; reproducible via its own RNG."""
    def __init__(self, amp_range=AMP_RANGE, phase_range=PHASE_RANGE,
                 x_range=(-5.0, 5.0), seed: int = 0):
        self.amp_range = amp_range
        self.phase_range = phase_range
        self.x_range = x_range
        self.rng = np.random.default_rng(seed)

    def sample_task(self, K: int, n_query: int = 100) -> dict:
        amplitude = float(self.rng.uniform(*self.amp_range))
        phase = float(self.rng.uniform(*self.phase_range))
        return make_sine_task(K=K, amplitude=amplitude, phase=phase,
                              x_range=self.x_range, n_query=n_query, rng=self.rng)

    def sample_batch(self, batch_size: int, K: int, n_query: int = 100) -> list:
        return [self.sample_task(K, n_query) for _ in range(batch_size)]

task_dist = TaskDistribution(seed=0)

def plot_task_grid(tasks: list, ncols: int = 3) -> None:
    import math
    n = len(tasks)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 2.6 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, t in enumerate(tasks):
        ax = axes[i]
        ax.plot(t["query_x"].numpy(), t["query_y"].numpy(), "g-")
        ax.scatter(t["support_x"].numpy(), t["support_y"].numpy(), c="k", s=18, zorder=5)
        ax.set_title(f"A={t['amplitude']:.2f}, phi={t['phase']:.2f}", fontsize=9)
    for j in range(n, len(axes)):  # hide unused axes
        axes[j].axis("off")
    fig.suptitle("Several tasks sampled from p(T)", y=1.02)
    fig.tight_layout(); plt.show()

def draw(K: int, amp_max: float) -> None:
    amp_max = max(amp_max, AMP_RANGE[0] + 0.1)  # avoid degenerate interval
    explore = TaskDistribution(amp_range=(AMP_RANGE[0], amp_max), seed=1)
    tasks = explore.sample_batch(6, K=K)
    plot_task_grid(tasks, ncols=3)

if WIDGETS_AVAILABLE:
    interact(draw,
             K=IntSlider(min=2, max=20, step=1, value=10, description="K (shots)"),
             amp_max=FloatSlider(min=1.0, max=5.0, step=0.5, value=5.0, description="amp max"))
else:
    print("ipywidgets unavailable — rendering a static grid at K=10, amp_max=5.0.")
    draw(10, 5.0)

# Sanity checks.
batch = task_dist.sample_batch(4, K=5)
assert len(batch) == 4 and batch[0]["support_x"].shape == (5, 1)
assert TaskDistribution(seed=0).sample_task(5)["amplitude"] == TaskDistribution(seed=0).sample_task(5)["amplitude"]
print("TaskDistribution ready; reproducible sampling confirmed.")

## Concept 3 — MAML: learning a good *initialization*

**Model-Agnostic Meta-Learning (MAML)** makes one specific choice for what is learnable: **`φ` = the network's initial parameters**. The idea is to find a starting point from which *any* new task can be reached in just a few gradient steps.

MAML has two nested loops:

**Inner loop (adaptation), per task.** Start from the shared initialization `φ`. Take a few SGD steps on the task's **support** set to get *task-adapted* weights `θ'`:

```
theta'  =  phi  -  alpha * grad_phi  L_support(phi)        # repeated for a few inner steps
```

**Outer loop (meta-update).** We want `θ'` to do well on that task's **query** set. So we update the *initialization* `φ` to minimize the post-adaptation query loss, **differentiating through the inner update**:

```
phi  <-  phi  -  beta * grad_phi  sum_tasks  L_query(theta'(phi))
```

The key twist: the meta-gradient flows *through* the adaptation step, so `φ` is optimized for *"where it ends up after adapting,"* not for where it starts.

> **First-order MAML (FOMAML)** drops the second-derivative term (treats `θ'` as if `φ` were constant during the inner step). It's far cheaper, numerically stable, and works almost as well — so it's our default for this demo. A small number of inner steps keeps everything fast on CPU.

The next cell implements all of this with a *functional* forward pass, so we can apply "fast weights" without mutating an `nn.Module` in place.

In [ ]:
# C08 — Implement and train MAML (first-order by default).

def functional_forward(x: torch.Tensor, params: list, hidden: int = 40) -> torch.Tensor:
    """Stateless MLP forward using a flat param list [W1,b1,W2,b2,W3,b3].
    Lets the inner loop apply fast weights without nn.Module in-place ops."""
    W1, b1, W2, b2, W3, b3 = params
    h = F.relu(F.linear(x, W1, b1))
    h = F.relu(F.linear(h, W2, b2))
    return F.linear(h, W3, b3)

def init_params() -> list:
    """Meta-parameters phi: a fresh MLPRegressor's params as a flat, grad-enabled list."""
    net = MLPRegressor().to(DEVICE)
    return [p.detach().clone().requires_grad_(True) for p in net.parameters()]

def inner_adapt(params: list, support_x, support_y, inner_steps: int,
                inner_lr: float = 0.01, create_graph: bool = False) -> list:
    """k SGD steps on the support set. create_graph=False => first-order MAML.
    inner_steps=0 returns a clone of the input params (no-op)."""
    fast_weights = [p for p in params]
    for _ in range(inner_steps):
        pred = functional_forward(support_x, fast_weights)
        loss = F.mse_loss(pred, support_y)
        grads = torch.autograd.grad(loss, fast_weights, create_graph=create_graph)
        fast_weights = [w - inner_lr * g for w, g in zip(fast_weights, grads)]
    if inner_steps == 0:
        fast_weights = [w.clone() for w in fast_weights]
    return fast_weights

def maml_meta_train(task_dist, meta_iters: int = 300, meta_batch: int = 4,
                    inner_steps: int = 1, inner_lr: float = 0.01,
                    meta_lr: float = 1e-3, K: int = 10) -> tuple:
    set_seed(0)
    phi = init_params()
    meta_opt = torch.optim.Adam(phi, lr=meta_lr)
    loss_curve = []
    for it in range(meta_iters):
        meta_opt.zero_grad()
        tasks = task_dist.sample_batch(meta_batch, K=K)
        meta_loss = 0.0
        try:
            for t in tasks:
                fast_weights = inner_adapt(phi, t["support_x"], t["support_y"],
                                           inner_steps=inner_steps, inner_lr=inner_lr,
                                           create_graph=False)
                q_pred = functional_forward(t["query_x"], fast_weights)
                meta_loss = meta_loss + F.mse_loss(q_pred, t["query_y"])
            meta_loss = meta_loss / len(tasks)
            meta_loss.backward()
            meta_opt.step()
        except RuntimeError as e:
            print(f"[meta-iter {it}] stopping early on RuntimeError: {e}")
            break
        loss_curve.append(meta_loss.item())
        if it % 50 == 0 or it == meta_iters - 1:
            print(f"meta-iter {it:4d}  meta-loss={loss_curve[-1]:.4f}")
    return phi, loss_curve

# Small, timeout-safe budget (meta_iters reduced to stay well under the Stage 5 cell timeout).
maml_init, maml_loss_curve = maml_meta_train(task_dist, meta_iters=100, meta_batch=4,
                                             inner_steps=1, inner_lr=0.01, meta_lr=1e-3, K=10)

plt.figure()
plt.plot(maml_loss_curve, alpha=0.35, label="meta-loss")
plt.plot(np.arange(len(moving_average(maml_loss_curve))), moving_average(maml_loss_curve),
         "b-", label="smoothed")
plt.xlabel("meta-iteration"); plt.ylabel("query MSE after adaptation")
plt.title("MAML meta-training loss"); plt.legend(); plt.show()

# Sanity checks.
assert len(maml_init) == len(list(MLPRegressor().parameters()))
assert functional_forward(torch.zeros(2, 1), maml_init).shape == (2, 1)
assert maml_loss_curve[-1] < maml_loss_curve[0]
print("MAML trained. meta-loss: {:.4f} -> {:.4f}".format(maml_loss_curve[0], maml_loss_curve[-1]))

In [ ]:
# C09 — Visualize fast adaptation on a FRESH held-out task.

set_seed(123)  # held-out eval seed, distinct from training
eval_dist = TaskDistribution(seed=999)
eval_task = eval_dist.sample_task(K=10)

def adapt_and_predict(n_steps: int):
    """Adapt maml_init for n_steps on the held-out support set; return (preds, query_mse)."""
    n_steps = max(int(n_steps), 0)
    fast_weights = inner_adapt(maml_init, eval_task["support_x"], eval_task["support_y"],
                               inner_steps=n_steps, create_graph=False)
    with torch.no_grad():
        preds = functional_forward(eval_task["query_x"], fast_weights)
        mse = F.mse_loss(preds, eval_task["query_y"]).item()
    return preds, mse

def plot_adapt(n_steps: int) -> None:
    preds, mse = adapt_and_predict(n_steps)
    plt.figure()
    plt.plot(eval_task["query_x"].numpy(), eval_task["query_y"].numpy(), "g-", label="true curve")
    plt.plot(eval_task["query_x"].numpy(), preds.numpy(), "b--", label=f"adapted ({n_steps} steps)")
    plt.scatter(eval_task["support_x"].numpy(), eval_task["support_y"].numpy(),
                c="k", zorder=5, label="support points")
    plt.title(f"Fast adaptation from MAML init: {n_steps} inner step(s), query MSE={mse:.3f}")
    plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

if WIDGETS_AVAILABLE:
    interact(plot_adapt, n_steps=IntSlider(min=0, max=10, step=1, value=1, description="inner steps"))
else:
    print("ipywidgets unavailable — static multi-panel adaptation figure.")
    steps_list = [0, 1, 3, 5, 10]
    fig, axes = plt.subplots(1, len(steps_list), figsize=(3.2 * len(steps_list), 3))
    for ax, s in zip(axes, steps_list):
        preds, mse = adapt_and_predict(s)
        ax.plot(eval_task["query_x"].numpy(), eval_task["query_y"].numpy(), "g-")
        ax.plot(eval_task["query_x"].numpy(), preds.numpy(), "b--")
        ax.scatter(eval_task["support_x"].numpy(), eval_task["support_y"].numpy(), c="k", s=16, zorder=5)
        ax.set_title(f"{s} steps\nMSE={mse:.2f}", fontsize=9)
    fig.suptitle("MAML init snaps onto a new task in a few steps", y=1.04)
    fig.tight_layout(); plt.show()
    for s in steps_list:
        print(f"  n_steps={s:2d}  query MSE={adapt_and_predict(s)[1]:.3f}")

mse0, mse5 = adapt_and_predict(0)[1], adapt_and_predict(5)[1]
print(f"Post-adaptation query MSE: 0 steps={mse0:.3f}  ->  5 steps={mse5:.3f}")
assert adapt_and_predict(5)[0].shape == eval_task["query_x"].shape

## Concept 4 — MAML vs. multi-task pre-training

A natural question: isn't this just **transfer learning**? Couldn't we train one network on *all* the tasks at once and fine-tune it? That's the **multi-task pre-training** baseline, and comparing it to MAML reveals what makes meta-learning special.

- **Multi-task pre-training** pools every task's data and finds parameters that minimize the *average* loss **right now**. The result is a compromise model — decent everywhere, optimal nowhere. After fine-tuning it can adapt, but it was never *optimized for adaptation*.
- **MAML** explicitly optimizes for *post-adaptation* performance. Its initialization may look mediocre before adaptation, but it sits at a point in weight space from which a few gradient steps reach an excellent task-specific solution.

Picture the loss landscapes of many tasks. Pre-training parks you at the point that's lowest *on average*. MAML parks you at the point that is **closest, in gradient-steps, to every task's individual minimum** — even if that point isn't especially low for any single task on its own.

> For our sine tasks the difference is stark: averaging over sine waves with all phases gives roughly a flat line (≈0), so a multi-task model adapts slowly. MAML, optimized to adapt, locks onto a new wave in a couple of steps.

The next two cells train the baseline under the **same compute budget** as MAML, then run a fair head-to-head comparison.

In [ ]:
# C11 — Multi-task pre-training baseline (same budget as MAML, NO inner-loop adaptation).

def pretrain_baseline(task_dist, iters: int = 300, batch: int = 4,
                      lr: float = 1e-3, K: int = 10) -> tuple:
    """Jointly minimize regression loss over many tasks' (support+query) data.
    Budget matches maml_meta_train (iters*batch identical) for a fair comparison."""
    set_seed(0)
    phi = init_params()  # SAME architecture & init scheme as MAML
    opt = torch.optim.Adam(phi, lr=lr)
    loss_curve = []
    for it in range(iters):
        opt.zero_grad()
        tasks = task_dist.sample_batch(batch, K=K)
        total = 0.0
        for t in tasks:
            all_x = torch.cat([t["support_x"], t["query_x"]], dim=0)
            all_y = torch.cat([t["support_y"], t["query_y"]], dim=0)
            total = total + F.mse_loss(functional_forward(all_x, phi), all_y)
        total = total / len(tasks)
        total.backward()
        opt.step()
        loss_curve.append(total.item())
        if it % 50 == 0 or it == iters - 1:
            print(f"pretrain-iter {it:4d}  loss={loss_curve[-1]:.4f}")
    return phi, loss_curve

# Match MAML's budget: same iters and batch as maml_meta_train above.
pretrained_init, pretrain_loss_curve = pretrain_baseline(task_dist, iters=100, batch=4,
                                                          lr=1e-3, K=10)

plt.figure()
plt.plot(pretrain_loss_curve, alpha=0.35, label="train loss")
plt.plot(np.arange(len(moving_average(pretrain_loss_curve))), moving_average(pretrain_loss_curve),
         "m-", label="smoothed")
plt.xlabel("iteration"); plt.ylabel("joint regression MSE")
plt.title("Multi-task pre-training baseline loss"); plt.legend(); plt.show()

# Sanity checks.
assert len(pretrained_init) == len(maml_init)
assert functional_forward(torch.zeros(2, 1), pretrained_init).shape == (2, 1)
assert pretrain_loss_curve[-1] < pretrain_loss_curve[0]
print("Baseline trained. loss: {:.4f} -> {:.4f}".format(pretrain_loss_curve[0], pretrain_loss_curve[-1]))

In [ ]:
# C12 — Head-to-head: MAML init vs. pre-trained init on fresh held-out tasks.

def eval_mse_vs_steps(init_params_list: list, tasks: list, max_steps: int) -> list:
    """Mean query MSE across tasks after s=0..max_steps inner adaptation steps."""
    means = []
    for s in range(max_steps + 1):
        per_task = []
        for t in tasks:
            fw = inner_adapt(init_params_list, t["support_x"], t["support_y"],
                             inner_steps=s, create_graph=False)
            with torch.no_grad():
                per_task.append(F.mse_loss(functional_forward(t["query_x"], fw), t["query_y"]).item())
        means.append(float(np.mean(per_task)))
    return means

def compare(inner_steps: int, n_eval_tasks: int) -> None:
    inner_steps = max(int(inner_steps), 0)
    n_eval_tasks = max(int(n_eval_tasks), 1)
    cmp_dist = TaskDistribution(seed=2024)  # held-out eval set, distinct from training
    tasks = cmp_dist.sample_batch(n_eval_tasks, K=10)

    maml_curve = eval_mse_vs_steps(maml_init, tasks, inner_steps)
    pre_curve = eval_mse_vs_steps(pretrained_init, tasks, inner_steps)

    # Panel (a): one example task, fitted curves after `inner_steps` steps.
    t0 = tasks[0]
    fw_m = inner_adapt(maml_init, t0["support_x"], t0["support_y"], inner_steps=inner_steps)
    fw_p = inner_adapt(pretrained_init, t0["support_x"], t0["support_y"], inner_steps=inner_steps)
    with torch.no_grad():
        pred_m = functional_forward(t0["query_x"], fw_m).numpy()
        pred_p = functional_forward(t0["query_x"], fw_p).numpy()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(t0["query_x"].numpy(), t0["query_y"].numpy(), "g-", label="true")
    ax1.plot(t0["query_x"].numpy(), pred_m, "b--", label="MAML")
    ax1.plot(t0["query_x"].numpy(), pred_p, "m:", label="pre-train")
    ax1.scatter(t0["support_x"].numpy(), t0["support_y"].numpy(), c="k", zorder=5, label="support")
    ax1.set_title(f"Example task after {inner_steps} step(s)"); ax1.set_xlabel("x"); ax1.legend()

    steps_axis = np.arange(inner_steps + 1)
    ax2.plot(steps_axis, maml_curve, "b-o", label="MAML")
    ax2.plot(steps_axis, pre_curve, "m-s", label="pre-train")
    ax2.set_xlabel("inner gradient steps"); ax2.set_ylabel("mean query MSE")
    ax2.set_title(f"Adaptation over {n_eval_tasks} held-out tasks"); ax2.legend()
    fig.tight_layout(); plt.show()

    print(f"Post-adaptation mean query MSE @ {inner_steps} steps:")
    print(f"  MAML        : {maml_curve[inner_steps]:.4f}")
    print(f"  Pre-training: {pre_curve[inner_steps]:.4f}")

if WIDGETS_AVAILABLE:
    interact(compare,
             inner_steps=IntSlider(min=0, max=10, step=1, value=5, description="inner steps"),
             n_eval_tasks=IntSlider(min=1, max=20, step=1, value=8, description="eval tasks"))
else:
    print("ipywidgets unavailable — static comparison at inner_steps=5, n_eval_tasks=8.")
    compare(5, 8)

# Sanity checks.
_tasks = TaskDistribution(seed=2024).sample_batch(8, K=10)
assert len(eval_mse_vs_steps(maml_init, _tasks, 5)) == 6
m5 = eval_mse_vs_steps(maml_init, _tasks, 5)[5]
p5 = eval_mse_vs_steps(pretrained_init, _tasks, 5)[5]
print(f"Check (tolerant): MAML@5={m5:.3f} vs pre-train@5={p5:.3f}")
assert m5 <= p5 + 1.0  # tolerance for seed variance

## Recap & where to go next

We walked one story end to end, all on a few-shot sine-regression sandbox that runs on CPU:

1. **The 3-step framework** — meta-learning is *function → loss → optimization*, but the learnable object is a learning algorithm `F_φ`. *Standard ML finds `f`; meta-learning finds the `F` that finds `f`.* (C03–C04)
2. **Episodic tasks** — each task splits into a **support** set (adapt) and a **query** set (evaluate), sampled from a task distribution `p(T)`; the same recipe scales to N-way K-shot Omniglot. (C05–C06)
3. **MAML** — learn an **initialization** via an inner adaptation loop and an outer meta-update that differentiates through it. We watched it **snap onto a brand-new task in a few gradient steps**. (C07–C09)
4. **MAML vs. pre-training** — an initialization optimized *to adapt* beats one optimized *to be good on average now*, head to head under an equal budget. (C10–C12)

**Deferred topics — natural next steps:**

- **MAML variants:** Reptile, first-order MAML (FOMAML), and ANIL (adapt only the head).
- **The full "what is learnable" taxonomy:** learnable optimizers, neural architecture search (NAS), data-augmentation policies, and sample reweighting — `φ` need not be the initialization.
- **Metric-based meta-learning:** Siamese, Prototypical, Matching, and Relation networks, which compare query examples to support examples instead of adapting weights.
- **Scale up:** move from sine regression to real **N-way K-shot image classification** on Omniglot / *mini*ImageNet.

The sine sandbox is deliberately tiny — swap in your own `make_sine_task`/`TaskDistribution` (or a classification task sampler) and the same MAML machinery carries straight over.